# 🌱 Plant Disease Detection AI - Part 2

## Advanced Features: Voice, Severity, Treatment & Complete Inference

This notebook continues from Part 1 and adds:
- **🎤 Voice Input Processing** (Hindi/Marathi)
- **📊 Disease Severity Estimation**
- **💊 Treatment Recommendations**
- **📈 Progress Tracking**
- **🔊 Text-to-Speech Responses**
- **🎯 Complete Multi-Modal Inference Pipeline**

### Prerequisites: Run Part 1 first or load pre-trained model

## 📋 Section 1: Load Pre-trained Model & Setup

In [ ]:
# Quick dataset setup using Google Drive (recommended for large files)
from google.colab import drive
import os

print("🚀 Setting up dataset access...")
print("Choose your preferred method:")
print("1. Google Drive (recommended for 4.5GB dataset)")
print("2. Direct upload (only for small samples)")
print("3. Kaggle API (if dataset is on Kaggle)")

USE_DRIVE = True  # Set to True for Google Drive method

if USE_DRIVE:
    print("📁 Mounting Google Drive...")
    drive.mount('/content/drive')
    
    # Update this path to your dataset location in Drive
    DATASET_PATH = '/content/drive/MyDrive/plant_disease_dataset/'
    print(f"✅ Dataset will be loaded from: {DATASET_PATH}")
    print("📝 Make sure you've uploaded your dataset to Google Drive first!")
else:
    print("📤 Using direct upload method...")
    DATASET_PATH = '/content/dataset/'

In [ ]:
# Import libraries (if not already imported from Part 1)
import tensorflow as tf
import numpy as np
import cv2
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
import pickle
import json
import sqlite3
from datetime import datetime
import os
import warnings
warnings.filterwarnings('ignore')

# Audio processing
import librosa
import speech_recognition as sr
from gtts import gTTS
from IPython.display import Audio, display
import io

# Text processing
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

print("✅ Libraries loaded successfully!")

In [ ]:
# Load pre-trained model and label encoder
try:
    # Try to load from Part 1
    model = tf.keras.models.load_model('plant_disease_model.h5')
    with open('label_encoder.pkl', 'rb') as f:
        label_encoder = pickle.load(f)
    print("✅ Loaded model and label encoder from Part 1")
except:
    print("⚠️ Model not found. Please run Part 1 first or upload pre-trained model.")
    print("For now, we'll create a mock setup for development...")
    
    # Mock setup for development
    from sklearn.preprocessing import LabelEncoder
    label_encoder = LabelEncoder()
    # Common Indian crop diseases
    mock_classes = [
        'Healthy', 'Leaf_Blight', 'Brown_Spot', 'Bacterial_Wilt',
        'Yellow_Leaf_Disease', 'Powdery_Mildew', 'Rust', 'Mosaic_Virus'
    ]
    label_encoder.fit(mock_classes)
    model = None  # Will be replaced with actual model

num_classes = len(label_encoder.classes_)
print(f"📋 Number of disease classes: {num_classes}")
print(f"Classes: {list(label_encoder.classes_)}")

## 🔬 Section 2: Disease Severity Estimation

In [ ]:
def estimate_disease_severity(image_path_or_array, disease_class):
    """
    Estimate disease severity using color-based segmentation
    Returns: severity_percentage, severity_level
    """
    # Load image
    if isinstance(image_path_or_array, str):
        image = cv2.imread(image_path_or_array)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    else:
        image = image_path_or_array.copy()
    
    # Resize for processing
    image = cv2.resize(image, (512, 512))
    
    # Convert to different color spaces for better disease detection
    hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
    lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
    
    # Define disease-specific color ranges (you can expand this)
    disease_colors = {
        'Leaf_Blight': {
            'brown_lower': np.array([10, 50, 20]),
            'brown_upper': np.array([20, 255, 200])
        },
        'Brown_Spot': {
            'brown_lower': np.array([5, 50, 20]),
            'brown_upper': np.array([25, 255, 150])
        },
        'Yellow_Leaf_Disease': {
            'yellow_lower': np.array([20, 100, 100]),
            'yellow_upper': np.array([30, 255, 255])
        },
        'Rust': {
            'rust_lower': np.array([5, 50, 50]),
            'rust_upper': np.array([15, 255, 200])
        }
    }
    
    # Default diseased area detection (works for most diseases)
    if disease_class in disease_colors:
        color_ranges = disease_colors[disease_class]
    else:
        # Generic diseased area detection (brown/yellow/dried areas)
        color_ranges = {
            'diseased_lower': np.array([10, 30, 30]),
            'diseased_upper': np.array([30, 255, 200])
        }
    
    # Create masks for diseased areas
    total_mask = np.zeros(hsv.shape[:2], dtype=np.uint8)
    
    for color_name, (lower, upper) in zip(
        [list(color_ranges.keys())[i] for i in range(0, len(color_ranges), 2)],
        [(list(color_ranges.values())[i], list(color_ranges.values())[i+1]) 
         for i in range(0, len(color_ranges), 2)]
    ):
        mask = cv2.inRange(hsv, lower, upper)
        total_mask = cv2.bitwise_or(total_mask, mask)
    
    # Clean up the mask
    kernel = np.ones((3,3), np.uint8)
    total_mask = cv2.morphologyEx(total_mask, cv2.MORPH_CLOSE, kernel)
    total_mask = cv2.morphologyEx(total_mask, cv2.MORPH_OPEN, kernel)
    
    # Calculate green/healthy areas (for comparison)
    green_lower = np.array([40, 40, 40])
    green_upper = np.array([80, 255, 255])
    green_mask = cv2.inRange(hsv, green_lower, green_upper)
    
    # Calculate percentages
    total_pixels = image.shape[0] * image.shape[1]
    diseased_pixels = cv2.countNonZero(total_mask)
    healthy_pixels = cv2.countNonZero(green_mask)
    
    # Calculate severity percentage
    if healthy_pixels + diseased_pixels > 0:
        severity_percentage = (diseased_pixels / (healthy_pixels + diseased_pixels)) * 100
    else:
        severity_percentage = 20  # Default moderate severity
    
    # Classify severity level
    if severity_percentage < 15:
        severity_level = "Mild"
    elif severity_percentage < 40:
        severity_level = "Medium"
    else:
        severity_level = "Severe"
    
    return severity_percentage, severity_level, total_mask

def visualize_severity(image, mask, severity_percentage, severity_level):
    """
    Visualize the disease severity detection
    """
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Original image
    axes[0].imshow(image)
    axes[0].set_title('Original Image')
    axes[0].axis('off')
    
    # Disease mask
    axes[1].imshow(mask, cmap='hot')
    axes[1].set_title('Disease Areas (Red = Diseased)')
    axes[1].axis('off')
    
    # Overlay
    overlay = image.copy()
    overlay[mask > 0] = [255, 0, 0]  # Highlight diseased areas in red
    combined = cv2.addWeighted(image, 0.7, overlay, 0.3, 0)
    axes[2].imshow(combined)
    axes[2].set_title(f'Severity: {severity_level} ({severity_percentage:.1f}%)')
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()

print("✅ Severity estimation functions created!")

## 🎤 Section 3: Voice Input Processing (Hindi/Marathi)

In [ ]:
# Setup speech recognition
recognizer = sr.Recognizer()

# Hindi/Marathi symptom keywords mapping
symptom_keywords = {
    # Hindi keywords
    'पत्ते': ['leaf', 'leaves'],
    'पीले': ['yellow', 'yellowing'],
    'भूरे': ['brown', 'browning'],
    'धब्बे': ['spots', 'patches'],
    'सूखे': ['dry', 'dried', 'wilting'],
    'कीड़े': ['insects', 'pests', 'bugs'],
    'बीमारी': ['disease', 'infection'],
    'पौधा': ['plant', 'crop'],
    'फसल': ['crop', 'harvest'],
    'गेहूं': ['wheat'],
    'धान': ['rice', 'paddy'],
    'मक्का': ['maize', 'corn'],
    'कपास': ['cotton'],
    'टमाटर': ['tomato'],
    'आलू': ['potato'],
    
    # Marathi keywords  
    'पाने': ['leaf', 'leaves'],
    'पिवळे': ['yellow'],
    'तपकिरी': ['brown'],
    'डाग': ['spots'],
    'कोरडे': ['dry'],
    'किडे': ['insects', 'pests'],
    'रोग': ['disease'],
    'झाड': ['plant'],
    'पिक': ['crop']
}

def process_voice_input(audio_file_path=None, duration=5):
    """
    Process voice input and extract symptoms
    """
    try:
        if audio_file_path:
            # Process uploaded audio file
            with sr.AudioFile(audio_file_path) as source:
                audio = recognizer.record(source)
        else:
            # Record from microphone (for local testing)
            with sr.Microphone() as source:
                print("🎤 Listening... Speak now in Hindi or Marathi")
                recognizer.adjust_for_ambient_noise(source)
                audio = recognizer.listen(source, timeout=duration)
        
        # Try to recognize speech in Hindi first, then English
        try:
            text_hi = recognizer.recognize_google(audio, language='hi-IN')
            print(f"🔊 Hindi: {text_hi}")
            recognized_text = text_hi
        except:
            try:
                text_mr = recognizer.recognize_google(audio, language='mr-IN')
                print(f"🔊 Marathi: {text_mr}")
                recognized_text = text_mr
            except:
                text_en = recognizer.recognize_google(audio, language='en-IN')
                print(f"🔊 English: {text_en}")
                recognized_text = text_en
        
        return recognized_text
    
    except Exception as e:
        print(f"❌ Error in voice recognition: {e}")
        return None

def extract_symptoms_from_text(text):
    """
    Extract symptom keywords from recognized text
    """
    if not text:
        return []
    
    text_lower = text.lower()
    detected_symptoms = []
    
    for hindi_word, english_equivalents in symptom_keywords.items():
        if hindi_word in text:
            detected_symptoms.extend(english_equivalents)
        
        for eng_word in english_equivalents:
            if eng_word in text_lower:
                detected_symptoms.append(eng_word)
    
    return list(set(detected_symptoms))  # Remove duplicates

def match_symptoms_to_disease(symptoms, predicted_disease):
    """
    Check if voice symptoms match with image-predicted disease
    """
    disease_symptom_map = {
        'Leaf_Blight': ['brown', 'spots', 'leaf', 'dry'],
        'Brown_Spot': ['brown', 'spots', 'patches'],
        'Yellow_Leaf_Disease': ['yellow', 'leaf', 'yellowing'],
        'Bacterial_Wilt': ['wilting', 'dry', 'brown'],
        'Powdery_Mildew': ['white', 'powder', 'leaf'],
        'Rust': ['rust', 'orange', 'spots'],
        'Mosaic_Virus': ['mosaic', 'pattern', 'yellow']
    }
    
    if predicted_disease in disease_symptom_map:
        expected_symptoms = disease_symptom_map[predicted_disease]
        matches = len(set(symptoms) & set(expected_symptoms))
        confidence_boost = min(matches * 0.15, 0.3)  # Max 30% boost
        return confidence_boost, matches
    
    return 0, 0

print("✅ Voice processing functions created!")

## 💊 Section 4: Treatment Database & Recommendations

In [ ]:
# Create comprehensive disease treatment database
treatment_database = {
    'Leaf_Blight': {
        'hindi_name': 'पत्ती झुलसा रोग',
        'description': 'Fungal disease causing brown spots and leaf death',
        'cure_steps': [
            'खेत में पानी भराव रोकें',
            'संक्रमित पत्तियां हटा दें',
            'कार्बेंडाजिम या मैंकोजेब का छिड़काव करें',
            '15 दिन बाद दोबारा छिड़काव करें',
            'फसल की निगरानी करते रहें'
        ],
        'treatments': [
            {'name': 'Carbendazim', 'cost': 600, 'effectiveness': 90},
            {'name': 'Mancozeb', 'cost': 400, 'effectiveness': 85},
            {'name': 'Neem Oil', 'cost': 200, 'effectiveness': 70},
            {'name': 'Copper Sulfate', 'cost': 300, 'effectiveness': 75}
        ],
        'yield_loss': {'Mild': 5, 'Medium': 20, 'Severe': 45}
    },
    'Brown_Spot': {
        'hindi_name': 'भूरे धब्बे का रोग',
        'description': 'Fungal disease with characteristic brown spots',
        'cure_steps': [
            'बीज उपचार करें',
            'खेत की सफाई रखें',
            'ट्राइकाइक्लाजोल का छिड़काव करें',
            'उर्वरक संतुलन बनाए रखें',
            'नियमित निरीक्षण करें'
        ],
        'treatments': [
            {'name': 'Tricyclazole', 'cost': 500, 'effectiveness': 88},
            {'name': 'Propiconazole', 'cost': 700, 'effectiveness': 92},
            {'name': 'Neem Oil', 'cost': 200, 'effectiveness': 65},
            {'name': 'Bordeaux Mixture', 'cost': 250, 'effectiveness': 70}
        ],
        'yield_loss': {'Mild': 8, 'Medium': 25, 'Severe': 50}
    },
    'Yellow_Leaf_Disease': {
        'hindi_name': 'पीली पत्ती रोग',
        'description': 'Viral disease causing yellowing of leaves',
        'cure_steps': [
            'संक्रमित पौधे हटा दें',
            'कीट नियंत्रण करें',
            'स्वस्थ बीज का उपयोग करें',
            'खेत में स्वच्छता बनाए रखें',
            'प्रतिरोधी किस्में लगाएं'
        ],
        'treatments': [
            {'name': 'Vector Control', 'cost': 400, 'effectiveness': 80},
            {'name': 'Imidacloprid', 'cost': 500, 'effectiveness': 85},
            {'name': 'Neem Oil', 'cost': 200, 'effectiveness': 60},
            {'name': 'Plant Removal', 'cost': 100, 'effectiveness': 95}
        ],
        'yield_loss': {'Mild': 15, 'Medium': 35, 'Severe': 70}
    },
    'Bacterial_Wilt': {
        'hindi_name': 'जीवाणु मुरझान रोग',
        'description': 'Bacterial infection causing plant wilting',
        'cure_steps': [
            'संक्रमित पौधे तुरंत हटाएं',
            'मिट्टी का उपचार करें',
            'जल निकासी सुधारें',
            'प्रतिरोधी किस्में उगाएं',
            'फसल चक्र अपनाएं'
        ],
        'treatments': [
            {'name': 'Streptomycin', 'cost': 800, 'effectiveness': 75},
            {'name': 'Copper Compounds', 'cost': 350, 'effectiveness': 70},
            {'name': 'Soil Solarization', 'cost': 150, 'effectiveness': 85},
            {'name': 'Biocontrol Agents', 'cost': 300, 'effectiveness': 80}
        ],
        'yield_loss': {'Mild': 20, 'Medium': 40, 'Severe': 80}
    },
    'Healthy': {
        'hindi_name': 'स्वस्थ पौधा',
        'description': 'Plant is healthy - no treatment needed',
        'cure_steps': [
            'नियमित देखभाल जारी रखें',
            'संतुलित उर्वरक दें',
            'उचित सिंचाई करें',
            'निवारक छिड़काव करें',
            'फसल की निगरानी करें'
        ],
        'treatments': [
            {'name': 'Preventive Neem', 'cost': 150, 'effectiveness': 90},
            {'name': 'Balanced Fertilizer', 'cost': 200, 'effectiveness': 95},
            {'name': 'Organic Compost', 'cost': 100, 'effectiveness': 85}
        ],
        'yield_loss': {'Mild': 0, 'Medium': 0, 'Severe': 0}
    }
}

def get_treatment_recommendations(disease, severity_level):
    """
    Get treatment recommendations for a disease
    """
    if disease not in treatment_database:
        return None
    
    disease_info = treatment_database[disease]
    treatments = disease_info['treatments']
    
    # Sort treatments by cost-effectiveness ratio
    for treatment in treatments:
        treatment['cost_effectiveness'] = treatment['effectiveness'] / treatment['cost']
    
    treatments_sorted = sorted(treatments, key=lambda x: x['cost_effectiveness'], reverse=True)
    
    # Get yield loss estimation
    yield_loss = disease_info['yield_loss'][severity_level]
    
    return {
        'disease_info': disease_info,
        'recommended_treatments': treatments_sorted,
        'yield_loss_percent': yield_loss
    }

print("✅ Treatment database created!")

## 🔊 Section 5: Text-to-Speech (Hindi Response)

In [ ]:
def text_to_speech_hindi(text, filename='response.mp3'):
    """
    Convert Hindi text to speech
    """
    try:
        tts = gTTS(text=text, lang='hi', slow=False)
        tts.save(filename)
        
        # Play audio in Colab
        display(Audio(filename, autoplay=True))
        
        print(f"🔊 Audio saved as: {filename}")
        return filename
    except Exception as e:
        print(f"❌ Error in TTS: {e}")
        return None

def create_diagnosis_response(disease, severity_level, treatments, yield_loss):
    """
    Create a comprehensive response in Hindi
    """
    if disease == 'Healthy':
        response = f"""आपका पौधा स्वस्थ है। कोई बीमारी नहीं मिली। 
नियमित देखभाल जारी रखें और निवारक उपाय करते रहें।"""
    else:
        disease_info = treatment_database[disease]
        hindi_name = disease_info['hindi_name']
        
        best_treatment = treatments[0]
        cheapest_effective = min([t for t in treatments if t['effectiveness'] > 75], 
                               key=lambda x: x['cost'], default=treatments[0])
        
        response = f"""आपके पौधे में {hindi_name} की बीमारी है। 
बीमारी की गंभीरता: {severity_level}
अगर इलाज नहीं किया तो {yield_loss}% फसल का नुकसान हो सकता है।

सबसे अच्छा इलाज: {best_treatment['name']} (₹{best_treatment['cost']})
सबसे सस्ता प्रभावी इलाज: {cheapest_effective['name']} (₹{cheapest_effective['cost']})

तुरंत उपचार शुरू करें और नियमित निगरानी करते रहें।"""
    
    return response

print("✅ Text-to-Speech functions created!")

## 📈 Section 6: Progress Tracking System

In [ ]:
# Initialize SQLite database for progress tracking
def init_progress_database():
    """
    Initialize SQLite database for tracking plant progress
    """
    conn = sqlite3.connect('plant_progress.db')
    cursor = conn.cursor()
    
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS plant_records (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            plant_id TEXT NOT NULL,
            timestamp DATETIME DEFAULT CURRENT_TIMESTAMP,
            disease TEXT,
            severity_level TEXT,
            severity_percentage REAL,
            confidence REAL,
            treatment_applied TEXT,
            image_path TEXT,
            notes TEXT
        )
    ''')
    
    conn.commit()
    conn.close()
    print("✅ Progress tracking database initialized!")

def save_diagnosis_record(plant_id, disease, severity_level, severity_percentage, 
                         confidence, treatment_applied="", image_path="", notes=""):
    """
    Save diagnosis record to database
    """
    conn = sqlite3.connect('plant_progress.db')
    cursor = conn.cursor()
    
    cursor.execute('''
        INSERT INTO plant_records 
        (plant_id, disease, severity_level, severity_percentage, 
         confidence, treatment_applied, image_path, notes)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    ''', (plant_id, disease, severity_level, severity_percentage, 
          confidence, treatment_applied, image_path, notes))
    
    conn.commit()
    conn.close()

def get_plant_progress(plant_id):
    """
    Get progress history for a specific plant
    """
    conn = sqlite3.connect('plant_progress.db')
    
    df = pd.read_sql_query(
        'SELECT * FROM plant_records WHERE plant_id = ? ORDER BY timestamp',
        conn, params=[plant_id]
    )
    
    conn.close()
    return df

def calculate_improvement(plant_id):
    """
    Calculate improvement percentage between last two records
    """
    df = get_plant_progress(plant_id)
    
    if len(df) < 2:
        return None, "Not enough data for comparison"
    
    latest = df.iloc[-1]
    previous = df.iloc[-2]
    
    # Calculate improvement in severity
    prev_severity = previous['severity_percentage']
    curr_severity = latest['severity_percentage']
    
    if prev_severity > 0:
        improvement = ((prev_severity - curr_severity) / prev_severity) * 100
    else:
        improvement = 0
    
    if improvement > 0:
        message = f"पिछली बार से {improvement:.1f}% सुधार हुआ है! 🌱"
    elif improvement < 0:
        message = f"पिछली बार से {abs(improvement):.1f}% बिगड़ाव हुआ है। तुरंत इलाज करें! ⚠️"
    else:
        message = "कोई खास बदलाव नहीं है। इलाज जारी रखें। 📊"
    
    return improvement, message

# Initialize database
init_progress_database()
print("✅ Progress tracking system ready!")

## 🎯 Section 7: Complete Multi-Modal Inference Pipeline

In [ ]:
def complete_plant_diagnosis(image_path, plant_id="default_plant", 
                           voice_input=None, save_record=True):
    """
    Complete multi-modal plant disease diagnosis pipeline
    
    Args:
        image_path: Path to plant image
        plant_id: Unique identifier for tracking
        voice_input: Audio file path or None for mic input
        save_record: Whether to save to progress database
    
    Returns:
        Complete diagnosis dictionary
    """
    print("🚀 Starting complete plant diagnosis...")
    
    # Step 1: Load and preprocess image
    image = cv2.imread(image_path)
    if image is None:
        return {"error": "Could not load image"}
    
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image_resized = cv2.resize(image_rgb, (224, 224))
    image_normalized = image_resized.astype(np.float32) / 255.0
    image_batch = np.expand_dims(image_normalized, axis=0)
    
    # Step 2: Image-based disease prediction
    if model is not None:
        predictions = model.predict(image_batch)
        predicted_class_idx = np.argmax(predictions[0])
        confidence = predictions[0][predicted_class_idx]
        predicted_disease = label_encoder.classes_[predicted_class_idx]
    else:
        # Mock prediction for demo
        predicted_disease = 'Leaf_Blight'
        confidence = 0.85
    
    print(f"📊 Image diagnosis: {predicted_disease} (confidence: {confidence:.2f})")
    
    # Step 3: Severity estimation
    severity_percentage, severity_level, severity_mask = estimate_disease_severity(
        image_rgb, predicted_disease
    )
    
    print(f"📈 Severity: {severity_level} ({severity_percentage:.1f}%)")
    
    # Step 4: Voice input processing (if provided)
    voice_symptoms = []
    voice_confidence_boost = 0
    
    if voice_input:
        print("🎤 Processing voice input...")
        recognized_text = process_voice_input(voice_input)
        if recognized_text:
            voice_symptoms = extract_symptoms_from_text(recognized_text)
            voice_confidence_boost, symptom_matches = match_symptoms_to_disease(
                voice_symptoms, predicted_disease
            )
            print(f"🗣️ Voice symptoms: {voice_symptoms}")
            print(f"✅ Symptom matches: {symptom_matches}, confidence boost: +{voice_confidence_boost:.2f}")
    
    # Step 5: Final confidence calculation
    final_confidence = min(confidence + voice_confidence_boost, 1.0)
    
    # Step 6: Treatment recommendations
    treatment_info = get_treatment_recommendations(predicted_disease, severity_level)
    
    # Step 7: Progress tracking
    improvement_percent, progress_message = calculate_improvement(plant_id)
    
    # Step 8: Create comprehensive response
    if treatment_info:
        hindi_response = create_diagnosis_response(
            predicted_disease, severity_level, 
            treatment_info['recommended_treatments'],
            treatment_info['yield_loss_percent']
        )
        
        if progress_message:
            hindi_response += f"\n\n{progress_message}"
    
    # Step 9: Save record to database
    if save_record:
        save_diagnosis_record(
            plant_id, predicted_disease, severity_level, 
            severity_percentage, final_confidence, 
            image_path=image_path
        )
    
    # Step 10: Generate voice response
    if treatment_info:
        audio_file = text_to_speech_hindi(hindi_response, f"diagnosis_{plant_id}.mp3")
    
    # Compile complete diagnosis
    diagnosis = {
        'plant_id': plant_id,
        'timestamp': datetime.now().isoformat(),
        'image_diagnosis': {
            'disease': predicted_disease,
            'confidence': float(confidence),
            'final_confidence': float(final_confidence)
        },
        'severity': {
            'level': severity_level,
            'percentage': float(severity_percentage)
        },
        'voice_analysis': {
            'symptoms_detected': voice_symptoms,
            'confidence_boost': float(voice_confidence_boost)
        },
        'treatment': treatment_info,
        'progress': {
            'improvement_percent': improvement_percent,
            'message': progress_message
        },
        'responses': {
            'hindi_text': hindi_response if treatment_info else "Error in treatment lookup",
            'audio_file': audio_file if treatment_info else None
        },
        'visualizations': {
            'original_image': image_rgb,
            'severity_mask': severity_mask
        }
    }
    
    return diagnosis

print("✅ Complete multi-modal diagnosis pipeline ready!")

## 🧪 Section 8: Demo & Testing

In [ ]:
# Demo function with sample images
def demo_diagnosis_system():
    """
    Demo the complete diagnosis system
    """
    print("🎬 Demo: Plant Disease Detection System")
    print("="*50)
    
    # For demo, we'll create a mock image (you should use real images)
    # Create a sample diseased leaf image
    sample_image = np.random.randint(0, 255, (300, 300, 3), dtype=np.uint8)
    # Add some brown spots to simulate disease
    cv2.circle(sample_image, (100, 100), 30, (139, 69, 19), -1)
    cv2.circle(sample_image, (200, 150), 25, (160, 82, 45), -1)
    cv2.circle(sample_image, (150, 200), 20, (139, 69, 19), -1)
    
    # Save sample image
    cv2.imwrite('sample_diseased_leaf.jpg', sample_image)
    
    print("📸 Sample diseased leaf image created")
    
    # Run diagnosis
    result = complete_plant_diagnosis(
        image_path='sample_diseased_leaf.jpg',
        plant_id='demo_plant_001',
        voice_input=None  # Set to audio file path if available
    )
    
    # Display results
    print("\n📋 DIAGNOSIS RESULTS:")
    print(f"Disease: {result['image_diagnosis']['disease']}")
    print(f"Confidence: {result['image_diagnosis']['final_confidence']:.2f}")
    print(f"Severity: {result['severity']['level']} ({result['severity']['percentage']:.1f}%)")
    
    if result['treatment']:
        print(f"\n💊 TREATMENT:")
        best_treatment = result['treatment']['recommended_treatments'][0]
        print(f"Recommended: {best_treatment['name']} (₹{best_treatment['cost']})")
        print(f"Yield Loss Risk: {result['treatment']['yield_loss_percent']}%")
    
    print(f"\n🗣️ HINDI RESPONSE:")
    print(result['responses']['hindi_text'])
    
    # Visualize severity
    if 'visualizations' in result:
        visualize_severity(
            result['visualizations']['original_image'],
            result['visualizations']['severity_mask'],
            result['severity']['percentage'],
            result['severity']['level']
        )
    
    return result

print("✅ Demo system ready!")
print("\n🚀 Run demo_diagnosis_system() to test the complete pipeline!")

In [ ]:
# Instructions for using with real data
print("""
🌱 PLANT DISEASE DETECTION SYSTEM - READY TO USE!
================================================================

📖 HOW TO USE:

1. **Upload your dataset:**
   - Use Google Drive method for 4.5GB dataset
   - Or upload to Kaggle and use Kaggle API

2. **Train model (if not done in Part 1):**
   - Run Part 1 notebook first
   - Or load pre-trained model

3. **Test with real images:**
   result = complete_plant_diagnosis(
       image_path='path/to/your/plant/image.jpg',
       plant_id='field_1_plant_1',
       voice_input='path/to/voice/file.wav'  # Optional
   )

4. **View progress over time:**
   progress_df = get_plant_progress('field_1_plant_1')
   print(progress_df)

🎯 FEATURES INCLUDED:
✅ EfficientNetB3 Image Classification
✅ Disease Severity Estimation
✅ Hindi/Marathi Voice Input
✅ Multi-modal Confidence Scoring
✅ Treatment Recommendations
✅ Cost-effective Pesticide Suggestions
✅ Yield Loss Estimation
✅ Progress Tracking Database
✅ Hindi Text-to-Speech Responses
✅ Complete Inference Pipeline

🚀 Ready for production use!
""")

In [ ]:
# Run the demo!
demo_result = demo_diagnosis_system()